In [48]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

### 01. Data Preprocessing

In [76]:
df = pd.read_excel("fifa_world_national_teams.xlsx")

df.rename(columns={
        "Goleiro": "is_goalkeeper",
        "Zagueiro": "is_defender",
        "Meio": "is_midfielder",
        "Atacante": "is_forward",
        "GK_reflexes": "gk_reflexes"
    }, 
    inplace=True
)

X = df.drop(columns=["id", "name", "full_name", "overall_rating"])
y = df["overall_rating"]

binary_cols = ["is_goalkeeper", "is_defender", "is_midfielder", "is_forward"]
X_bin = X[binary_cols]

cat_cols = ["nationality", "national_team", "club_team"]
X_cat = X[cat_cols]

X_num = X.drop(columns=binary_cols + cat_cols)

X_cat_dummies = pd.get_dummies(X_cat, drop_first=True, dtype=float)

scaler = StandardScaler()
X_num_norm = scaler.fit_transform(X_num)
X_num_scaled = pd.DataFrame(X_num_norm, columns=X_num.columns, index=X.index)

X = pd.concat([X_num_scaled, X_cat_dummies, X_bin], axis=1)

In [77]:
# Separa 80% para treino e 20% para teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Adiciona a constante (intercepto) em ambos os conjuntos
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

### 02. Model Training

In [53]:
model = sm.OLS(y_train, X_train_sm)

train_results = model.fit()

print(train_results.summary())

                            OLS Regression Results                            
Dep. Variable:         overall_rating   R-squared:                       0.953
Model:                            OLS   Adj. R-squared:                  0.909
Method:                 Least Squares   F-statistic:                     21.62
Date:                Sun, 10 May 2026   Prob (F-statistic):          1.05e-115
Time:                        19:40:12   Log-Likelihood:                -967.72
No. Observations:                 574   AIC:                             2489.
Df Residuals:                     297   BIC:                             3695.
Df Model:                         276                                         
Covariance Type:            nonrobust                                         
                                              coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

In [79]:
pvalue_threshold = 0.05
X_train_sm_ = X_train_sm.copy()

X_train_sm_.drop(list(X_train_sm.filter(regex = 'nationality_')), axis = 1, inplace = True)

while True:
    model_fit = sm.OLS(y_train, X_train_sm_).fit()

    max_pvalue = model_fit.pvalues.max() 

    if  max_pvalue > pvalue_threshold:
        idx_max_pvalue = model_fit.pvalues.idxmax()
        print(f"INFO | Max p-value ({max_pvalue}) exceed threshold ({pvalue_threshold}) | removing '{idx_max_pvalue}' column")

        X_train_sm_.drop(columns=idx_max_pvalue, inplace=True)
    else:
        print(f"INFO | Max p-value ({max_pvalue}) within the threshold ({pvalue_threshold}) | Backwise step finish.")
        break

print(model_fit.summary())

INFO | Max p-value (0.9984902906105283) exceed threshold (0.05) | removing 'club_team_WisÅ‚a KrakÃ³w' column
INFO | Max p-value (0.9880408946441724) exceed threshold (0.05) | removing 'club_team_Hull City' column
INFO | Max p-value (0.9937459791969425) exceed threshold (0.05) | removing 'club_team_Sheffield Wednesday' column
INFO | Max p-value (0.9748559027047252) exceed threshold (0.05) | removing 'club_team_FC Schalke 04' column
INFO | Max p-value (0.9788623853415025) exceed threshold (0.05) | removing 'club_team_Dinamo Zagreb' column
INFO | Max p-value (0.973475072201675) exceed threshold (0.05) | removing 'club_team_En Avant de Guingamp' column
INFO | Max p-value (0.9753340176013843) exceed threshold (0.05) | removing 'club_team_KV Oostende' column
INFO | Max p-value (0.9725511762375934) exceed threshold (0.05) | removing 'club_team_Leicester City' column
INFO | Max p-value (0.9788255621131192) exceed threshold (0.05) | removing 'club_team_LOSC Lille' column
INFO | Max p-value (0.9

### 03. Model Evaluation

In [81]:
X_train_sm_

,const,value_euro,age,international_reputation(1-5),skill_moves(1-5),club_rating,heading_accuracy,short_passing,gk_reflexes,national_team_Brazil,...,club_team_PogoÅ„ Szczecin,club_team_Reading,club_team_Rizespor,club_team_SV Werder Bremen,club_team_Santos,club_team_Scunthorpe United,club_team_Torino,club_team_VfL Wolfsburg,club_team_Wolverhampton Wanderers,is_goalkeeper
56,1.0,0.301809,-0.436468,0.367313,1.274579,0.190621,0.795395,-0.273085,-0.151419,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
204,1.0,0.449688,0.121155,0.367313,0.223893,0.950988,1.195881,0.706637,-0.151419,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
319,1.0,-0.526312,-0.715279,-0.754945,-1.877479,-0.569746,-2.258313,-2.428475,2.549677,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
529,1.0,-0.727428,0.121155,-0.754945,-1.877479,-0.379654,-2.158191,-2.689734,2.504659,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
24,1.0,-0.715597,1.794022,-0.754945,-0.826793,-1.710297,0.444969,0.118804,-0.466547,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,1.0,-0.662361,2.351645,-0.754945,-1.877479,-0.569746,-2.458556,-3.146938,2.459640,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
106,1.0,-0.715597,-0.994090,-0.754945,-1.877479,-1.140021,-2.158191,-1.775326,2.684732,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
270,1.0,0.035627,-1.272902,-0.754945,0.223893,0.000530,0.695273,0.641323,-0.151419,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
435,1.0,-0.437585,2.072834,0.367313,1.274579,0.190621,0.645212,0.314748,-0.151419,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
